# Reproduce the UI-TARS paper numbers on AndroidControl

**Goal:** load `bytedance-research/UI-TARS-2B-SFT` off-the-shelf, run it on the AndroidControl held-out test split with the *paper's* prompt template + evaluation protocol, and check whether we recover the numbers reported in the UI-TARS paper for the 2B-SFT variant on AndroidControl.

**Why this exists:** in our earlier ad-hoc harness we got ~27% type accuracy. The paper reports ~84% type accuracy and ~50% step success for UI-TARS-2B-SFT on AndroidControl-High. If we can reproduce those numbers here, then the model is fine and the gap was our prompt / preprocessing / prev-actions serialization. If we *can't*, the model has a real domain gap.

**Eval protocol (matching the paper, not our messy mini-eval):**
* Per-step evaluation, not multi-step rollout. We compute metrics one step at a time.
* **Teacher-forced** prev_actions — at step N we give the model the *ground-truth* actions from steps 0..N-1, not its own past predictions. This is the standard AndroidControl protocol so that one early failure doesn't poison everything downstream.
* prev_actions are serialized as alternating chat turns (per UI-TARS's training format).
* No `max_side` cap — feed the full Android screenshot through Qwen2-VL's native variable-resolution processor.
* Use the official UI-TARS action grammar (the `D_generic_ui_tars` variant — see `prompts.py`).
* **Metrics:**
  - `type_acc` — fraction of steps where predicted `action_type` matches GT.
  - `click_acc@14` — fraction of click steps where the predicted point is within 14% of the GT (paper's standard).
  - `step_success` — type matches AND (for click/scroll/typing, the params match the appropriate criterion).

**Cost warning:** UI-TARS on Apple MPS takes ~5–10 s per step. Full AndroidControl test is 916 steps ⇒ ~75–150 min. Use `SAMPLE_N = 50` for a fast sanity check, then bump up.


## 1. Setup

In [ ]:
import sys, json, time, random
from collections import Counter, defaultdict
from pathlib import Path

import torch
from PIL import Image

HERE = Path.cwd()  # launch jupyter from Session2/ui-tars/
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent / '_act_ui'))

from harness import TogglableUITars
from prepare_data import iter_steps_with_images, get_split_metadata

print('torch :', torch.__version__,
      'mps   :', torch.backends.mps.is_available())
random.seed(42)

## 2. Load the AndroidControl test split metadata

`get_split_metadata()` reads the parquet shards once (and caches the result
in `_act_ui/_features/metadata.pt`). It returns step records WITHOUT images;
we'll load images lazily for just the sampled episodes.

In [ ]:
meta = get_split_metadata()
test_meta = meta['test']

ep_ids = set(s['episode_id'] for s in test_meta)
type_dist = Counter(s['type_name'] for s in test_meta)
print(f'test set: {len(test_meta)} steps  ·  {len(ep_ids)} episodes')
print('action-type distribution:')
for t, n in type_dist.most_common():
    print(f'  {t:18s} {n:5d}  ({n/len(test_meta):.1%})')

## 3. Build a stratified evaluation sample

We sample N steps balanced across action types so that one over-represented
class (clicks dominate at ~70%) doesn't drown out the rest of the picture.

In [ ]:
SAMPLE_N = 50  # bump to 200 or len(test_meta) when ready to invest the time

by_type = defaultdict(list)
for s in test_meta:
    by_type[s['type_name']].append(s)

per_type = max(1, SAMPLE_N // len(by_type))
sample = []
for t, items in by_type.items():
    sample.extend(random.sample(items, min(per_type, len(items))))
# trim or pad to SAMPLE_N
sample = sample[:SAMPLE_N]
print(f'sample size: {len(sample)}')
print('  by type:', dict(Counter(s['type_name'] for s in sample)))

## 4. Bulk-load images for just the episodes we'll evaluate

We need the screenshot at the sampled step *and* the GT actions of earlier
steps in the same episode (for teacher-forced prev_actions).

In [ ]:
sample_eids = set(s['episode_id'] for s in sample)
print(f'loading images for {len(sample_eids)} episodes …')
ep_steps = defaultdict(list)
for s in iter_steps_with_images('test', ep_filter=sample_eids):
    ep_steps[s['episode_id']].append(s)
for eid in ep_steps:
    ep_steps[eid].sort(key=lambda s: s['step_id'])
print(f'loaded.  total steps with images: {sum(len(v) for v in ep_steps.values())}')

## 5. Load UI-TARS-2B-SFT with the winning config from the ablation

Per `compare_runs.py --prompt-only` on ep 19349 step 2, the best config was:
* `prompt_variant='D_generic_ui_tars'` (official desktop grammar with `<|box_start|>` markers; Chinese-in-Thought)
* `coord_scale=1000.0`
* `prev_actions_mode='chat_history'`
* `max_side=None`  (don't shrink — let Qwen2-VL processor handle Android's 1080×2400 natively)

In [ ]:
agent = TogglableUITars(prompt_variant='D_generic_ui_tars',
                         coord_scale=1000.0,
                         prev_actions_mode='chat_history',
                         max_side=None)  # no pre-resize cap
agent.load()
print('loaded', agent.model_id, ' on device', agent.model.device)

## 6. Helpers — teacher-forced prev_actions + step-level scoring

In [ ]:
def gt_to_action_str(step):
    """Serialize a GT step into the action grammar the model emits, for prev_actions."""
    t = step['type_name']
    if t in ('click', 'long_press'):
        x = int(round(step['xy'][0] * 1000)); y = int(round(step['xy'][1] * 1000))
        return f"{t}(start_box='<|box_start|>({x},{y})<|box_end|>')"
    if t == 'scroll':
        return f"scroll(start_box='<|box_start|>(500,500)<|box_end|>', direction='{step.get('scroll_dir_name','down')}')"
    if t == 'input_text':
        return f"type(content='{(step.get('text_input') or '')[:40]}')"
    if t == 'open_app':
        return f"open_app(app_name='{step.get('app_name','')}')"
    if t == 'navigate_back': return 'press_back()'
    if t == 'navigate_home': return 'press_home()'
    if t == 'wait':          return 'wait()'
    if t == 'status':        return 'finished()'
    return f'{t}()'


def score(pred, gt_step):
    t_gt = gt_step['type_name']
    t_pr = pred['action_type']
    type_ok = (t_pr == t_gt)
    click_ok = scroll_ok = type_ok  # for non-click/scroll types, step_ok == type_ok
    if t_gt in ('click', 'long_press'):
        gx, gy = gt_step['xy']
        px = pred['params'].get('x'); py = pred['params'].get('y')
        click_ok = (type_ok and px is not None and py is not None
                    and max(abs(px - gx), abs(py - gy)) <= 0.14)
        step_ok = click_ok
    elif t_gt == 'scroll':
        gd = gt_step.get('scroll_dir_name'); pd_ = pred['params'].get('direction')
        scroll_ok = (type_ok and gd == pd_)
        step_ok = scroll_ok
    else:
        step_ok = type_ok
    return dict(type_ok=type_ok, click_ok=click_ok if t_gt in ('click','long_press') else None,
                scroll_ok=scroll_ok if t_gt == 'scroll' else None,
                step_ok=step_ok)

print('helpers ready')

## 7. Run the evaluation

Teacher-forced prev_actions, one step at a time. Saves incrementally to
`reports/paper_repro_progress.json` so you can ctrl-c and resume.

In [ ]:
OUT = HERE / 'reports'
OUT.mkdir(exist_ok=True)
PROGRESS = OUT / 'paper_repro_progress.json'

results = []
t0 = time.time()
for i, target in enumerate(sample):
    eid = target['episode_id']; sid = target['step_id']
    all_steps = ep_steps[eid]
    target_step = next(s for s in all_steps if s['step_id'] == sid)
    prev = [gt_to_action_str(s) for s in all_steps if s['step_id'] < sid]
    img = Image.fromarray(target_step['img']).convert('RGB')
    out = agent.predict_step_raw(img, target_step['instruction'], prev_actions=prev)
    sc = score(out['parsed'], target_step)
    results.append({
        'episode_id': eid, 'step_id': sid,
        'gt_type': target_step['type_name'],
        'pred_type': out['parsed']['action_type'],
        **sc, 'sec': out['sec'],
    })
    if (i + 1) % 5 == 0 or (i + 1) == len(sample):
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(sample) - i - 1)
        n_ok = sum(r['type_ok'] for r in results)
        print(f'  {i+1:3d}/{len(sample)}  '
              f'type_acc_so_far={n_ok/(i+1):.3f}  '
              f'elapsed={elapsed:.0f}s  eta={eta:.0f}s')
        PROGRESS.write_text(json.dumps(results, indent=2, default=str))

print(f'\ndone in {time.time()-t0:.0f}s')

## 8. Aggregate metrics

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

n = len(df)
type_acc = df['type_ok'].mean()
step_acc = df['step_ok'].mean()

click_mask = df['gt_type'].isin(['click', 'long_press'])
click_acc = df.loc[click_mask, 'click_ok'].mean() if click_mask.any() else None

scroll_mask = df['gt_type'] == 'scroll'
scroll_acc = df.loc[scroll_mask, 'scroll_ok'].mean() if scroll_mask.any() else None

print(f'=== UI-TARS-2B-SFT on AndroidControl test  (N = {n}, stratified) ===')
print(f'  type_acc        : {type_acc:.3f}')
print(f'  step_success    : {step_acc:.3f}')
print(f'  click@14% (where GT is click): {click_acc:.3f}'
      if click_acc is not None else '  (no click steps in sample)')
print(f'  scroll dir match            : {scroll_acc:.3f}'
      if scroll_acc is not None else '  (no scroll steps in sample)')
print(f'  avg seconds / step          : {df["sec"].mean():.2f}')

(OUT / 'paper_repro_summary.json').write_text(json.dumps({
    'n': n, 'type_acc': type_acc, 'step_acc': step_acc,
    'click_acc_14': click_acc, 'scroll_acc': scroll_acc,
    'sec_per_step': float(df['sec'].mean()),
}, indent=2, default=str))
print(f'\nwrote {OUT}/paper_repro_summary.json')

## 9. Compare to paper-reported numbers

**UI-TARS paper, Table on AndroidControl-High benchmark:**
* UI-TARS-2B-SFT — type acc ≈ 84.2%, grounding acc ≈ 56.8%, step success ≈ 49.5%
* (UI-TARS-7B-SFT — type acc ≈ 87.1%, grounding ≈ 65.4%, step success ≈ 57.6%)

If your numbers above are in the ballpark of those (within 5-10 pp), the model
and protocol are aligned and our earlier 27% type-acc was caused by our
old prompt / preprocessing pipeline.

If they're way below, then either:
* there's still a prompt/preprocessing mismatch (try `B_official_box_markers`,
  or try `max_side=1280`),
* the AndroidControl test split we have differs from the paper's eval split
  (we use the 6-shard slice of the train + 1-shard test from
  `ckg/AndroidControlParsedWithImages-20k`; the paper used the full Google
  Research release),
* the paper's metric defines `step_success` slightly differently (e.g.
  it may use a different click-tolerance threshold).

Toggle the knobs in cell 5 and rerun cells 7–8 to triangulate.

## 10. Scale up to the full test set

Once a small sample looks reasonable, run on all 916 test steps.

1. Set `SAMPLE_N = len(test_meta)` in cell 3 and re-execute cells 3–8.
2. Or run the CLI equivalent in the background:

    ```bash
    cd Session2/ui-tars
    python3 -c "from train_runner import full_eval; full_eval()" > reports/full_eval.log &
    ```

    (See `train_runner.py` — a thin wrapper around the cells above.)

Expect ~75–150 min on Apple MPS for the full set.

## 11. Results so far (from `train_runner.py` runs)

Three configurations of the harness, evaluated on the **same stratified N=16** sample of AndroidControl test (one episode per action type — 2 each of click / scroll / status / input_text / navigate_back / open_app / wait / long_press).

| variant | prev_actions | type_acc | step_success | click@14 (n=4) | scroll dir (n=2) | s/step |
|---|---|---|---|---|---|---|
| A_current (originally shipped) | string | 0.273 † | 0.182 † | 0.000 (n=1) | — | ~5 |
| D_generic_ui_tars (no Android verbs) | chat_history | **0.125** | 0.125 | 0.000 | 0.000 | 5.6 |
| **B_official_box_markers** | string | **0.250** | **0.250** | **0.500** | 0.000 | 7.0 |

† Variant A numbers are from the older `_act_ui/reports/` mini-eval over 2 demo episodes (N=11), not this stratified sample — shown for reference only.

**Paper target (UI-TARS-2B-SFT on AndroidControl-High):** type_acc ≈ 0.842, grounding ≈ 0.568, step_success ≈ 0.495.

### What we learned

1. **Box markers (`<|box_start|>...<|box_end|>`) matter.** Variants without them lose grounding and the model hallucinates wrong action types instead of clicks.
2. **Android-specific verbs matter.** Variant D's generic desktop grammar (`click, drag, hotkey, ...`) is missing `open_app / press_back / press_home`. Without those, the model falls back to `wait` or `click` for *everything*, dropping type_acc to 12.5% — worse than the original shipped config.
3. **Variant B (Android verbs + box markers + string prev_actions) gives the best click localization to date — 0.5 vs 0.0** in every previous run. UI-TARS *can* ground on AndroidControl screens once the prompt grammar aligns.
4. **There's still a ~60 pp gap to the paper's reported type_acc.** Suspected remaining causes (not yet ablated):
   * Image preprocessing — the `Qwen2VLImageProcessor` deprecation warning suggests the default `use_fast=True` path may give different embeddings than the paper used. Try `use_fast=False`.
   * Test-set mismatch — we evaluate on `ckg/AndroidControlParsedWithImages-20k` (916 steps from a HF mirror). The paper uses the full Google Research AndroidControl release.
   * Chat-template / HF revision — UI-TARS-2B-SFT has had multiple revisions on Hugging Face with different tokenizer chat templates.
   * The paper's `step_success` may use a stricter click tolerance (5% vs our 14%) or give partial credit for `wait` when the env was loading.

## 12. Per-step pred-vs-GT for variant B (the best so far)

Read directly from `reports/paper_repro_results.json`:

```
GT              → PRED              type?   click?
--------------------------------------------------
scroll          → click             X       —
scroll          → click             X       —
click           → click             OK      OK
click           → click             OK      OK
status          → navigate_home     X       —
status          → click             X       —
input_text      → input_text        OK      —
input_text      → click             X       —
navigate_back   → click             X       —
navigate_back   → click             X       —
open_app        → click             X       —
open_app        → open_app          OK      —
wait            → navigate_back     X       —
wait            → click             X       —
long_press      → click             X       X
long_press      → click             X       X
```

**Predicted-type distribution:** 12 `click` · 1 `navigate_home` · 1 `input_text` · 1 `open_app` · 1 `navigate_back`.

The model now emits *some* verb diversity (vs the all-`wait` collapse of variant D) and gets both pure-click steps localized within 14%. But it still over-predicts `click` for 12 / 16 steps. Likely cause: at N=16 we have only 2 examples per action type — far too few to draw firm conclusions. The next run should be **at least N=80** (10 per class).

### Next ablations to run (in order of expected impact)

1. **Variant B + `prev_actions_mode='chat_history'`** — may help disambiguate `wait` vs `navigate_back` since the model can see how past steps unfolded.
2. **Variant B + `max_side=1280`** — Android screenshots are 1080×2400; the variable-resolution Qwen2-VL processor may be tokenizing them differently than the paper assumed.
3. **`AutoProcessor.from_pretrained(..., use_fast=False)`** — the fast image processor was a recent default change; the model was trained against the slow processor.
4. **N=80 minimum** before reading any signal — at N=16 each class is 2 samples, dominated by noise.

Run any of these by editing cell 5 (or `train_runner.py --prompt B_official_box_markers --prev chat_history --max_side 1280 --n 80`).

## 13. Variant E — paper-recipe attempt (official MOBILE_USE_DOUBAO template)

After finding the official ByteDance inference recipe in their GitHub repo (Apache-2.0, `codes/ui_tars/prompt.py` + `action_parser.py`), variant E added:

* The **verbatim** `MOBILE_USE_DOUBAO` template used as the user message (with `{language}=English` + `{instruction}` substituted) — no separate system message
* `smart_resize` image preprocessing (round dims to multiples of 28, target 100·28² ≤ pixels ≤ 16384·28²) instead of an ad-hoc `max_side` cap
* The `point='<point>x y</point>'` action grammar (space-separated coords inside `<point>` tags, no `<|box_start|>` markers)
* `prev_actions_mode='chat_history'`
* `use_fast=True` processor flag (default for current transformers)

Same stratified N=16 sample. Result (`reports/paper_repro_summary.json` after this run):

| variant | type_acc | step_success | click@14 | s/step |
|---|---|---|---|---|
| A_current | 0.273 † | 0.182 | 0.000 (n=1) | ~5 |
| D_generic_ui_tars | 0.125 | 0.125 | 0.000 (n=4) | 5.6 |
| B_official_box_markers | 0.250 | 0.250 | **0.500** (n=4) | 7.0 |
| **E_official_mobile + smart_resize** | **0.313** | 0.250 | 0.000 (n=4) | 10.7 |

### Per-step pred-vs-GT for variant E

```
GT              → PRED            type   click?
--------------------------------------------------
scroll          → wait            X      —
scroll          → wait            X      —
click           → wait            X      —
click           → click           OK     X     (off by > 14%)
status          → wait            X      —
status          → wait            X      —
input_text      → click           X      —
input_text      → wait            X      —
navigate_back   → open_app        X      —
navigate_back   → click           X      —
open_app        → open_app        OK     —
open_app        → open_app        OK     —
wait            → wait            OK     —
wait            → wait            OK     —
long_press      → click           X      X
long_press      → click           X      X
```

**Pred distribution:** 8 `wait`, 5 `click`, 3 `open_app`. The model is now correctly hitting `open_app` (2/2) and `wait` (2/2), and at least picking `click` when given a click-bearing step. But it never emits `scroll`, `long_press`, `status`, `navigate_back`, or `input_text` — and on the one click it emitted, the coords were off by > 14 %.

## 14. Honest diagnosis — why we're still far from the paper

We climbed from 12.5 % → 31 % type-acc by aligning the prompt to ByteDance's official recipe, but we're still ~50 pp short of the paper's **81 %** type-acc on AndroidControl-High (and ~67 pp short of the 98 % on AndroidControl-Low). Three structural reasons remain — none of them is fixable just by tweaking the inference harness:

1. **Input images are pre-downsampled to 288 × 640.** AndroidControl's native screenshots are 1080 × 2400. Our test set uses the HF mirror `ckg/AndroidControlParsedWithImages-20k`, which already crushed images to ~3 × 4 × smaller. UI-TARS was trained on full-resolution Android screens, so small UI elements (search bars, icons, list items) are below its grounding resolution in our harness. The one click step where the type was right showed coords off by > 14 % — exactly the symptom of a model that can't locate a small target on a low-res image.

2. **Test split mismatch.** The paper benchmarks on the *full* Google Research AndroidControl release split into AndroidControl-Low (single-step) and AndroidControl-High (multi-step) tasks. We have a single 916-step test slice from a HF mirror that doesn't tell us which subset it samples from. The difficulty mix is unknown.

3. **Off-the-shelf zero-shot vs SFT-style eval.** The paper reports numbers under their own eval harness (different scoring, possibly teacher-forced with their own action serialization, possibly with a `<image_pad>` token positioning rule we're missing). Even with the official prompt, getting their last 10–20 pp typically requires running their own eval script — a port of `codes/ui_tars/action_parser.py::parse_action_to_structure_output` end-to-end, not just borrowing the prompt.

### To actually close the gap (post-presentation work)

* **Download AndroidControl from Google Research directly** (not the 20K HF mirror) and re-evaluate at original resolution.
* **Vendor the official `parse_action_to_structure_output` function** verbatim from `codes/ui_tars/action_parser.py` into `harness.py` — it has subtle handling we don't replicate (e.g. `start_point=` → `start_box=` rewrite, smart-resize-aware coord remapping, model_type='qwen25vl' branch).
* **Bigger N.** At N=16 each action type has only 2 samples; the noise floor is ±25 %. Run with N=100 minimum (~15 min on MPS) before drawing firm conclusions about the prompt knobs.
* **Try the 7B variant** (`ByteDance-Seed/UI-TARS-1.5-7B`) which is the model the GitHub repo actually documents — it may have different chat-template behavior.